# Mutual Fund Data Cleaning & Star Schema Loading

This notebook implements the data cleaning and validation routines for the Bluestock Mutual Fund Analytics Capstone, followed by loading the datasets into a SQLite star schema database.

### Objectives:
1. **Clean `nav_history.csv`**: Parse dates, sort, forward-fill missing NAVs for weekends/holidays, remove duplicates, validate `NAV > 0`.
2. **Clean `investor_transactions.csv`**: Standardize transaction types, validate `amount > 0`, fix date formats, validate KYC status.
3. **Clean `scheme_performance.csv`**: Validate numeric return values, flag mathematical anomalies, validate expense ratio ranges (0.1% - 2.5%).
4. **Load into SQLite Star Schema**: Populate database tables with referential integrity constraints.

In [1]:
# Import required libraries
import os
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
from sqlalchemy import create_engine, text

project_root = Path('..')
db_path = project_root / 'data' / 'db' / 'bluestock_mf.db'
processed_dir = project_root / 'data' / 'processed'
raw_dir = project_root / 'data' / 'raw'

## 1. Inspect Cleaning Logic & Results
Let's look at samples of the cleaned datasets exported by our cleaning modules.

In [2]:
# Load a sample of cleaned NAV history
df_nav = pd.read_csv(processed_dir / '02_nav_history.csv')
print(f"Cleaned NAV History Shape: {df_nav.shape}")
df_nav.head(5)

Cleaned NAV History Shape: (64320, 3)


,date,amfi_code,nav
0,2022-01-03,100016,520.4608
1,2022-01-04,100016,515.0971
2,2022-01-05,100016,521.7239
3,2022-01-06,100016,515.7880
4,2022-01-07,100016,515.1639


In [3]:
# Load a sample of cleaned investor transactions
df_tx = pd.read_csv(processed_dir / '08_investor_transactions.csv')
print(f"Cleaned Investor Transactions Shape: {df_tx.shape}")
df_tx.head(5)

Cleaned Investor Transactions Shape: (32778, 13)


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [4]:
# Load a sample of cleaned scheme performance
df_perf = pd.read_csv(processed_dir / '07_scheme_performance.csv')
print(f"Cleaned Scheme Performance Shape: {df_perf.shape}")
print(f"Anomalous schemes flagged: {df_perf['is_anomalous'].sum()}")
df_perf.head(5)

Cleaned Scheme Performance Shape: (40, 20)
Anomalous schemes flagged: 3


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade,is_anomalous
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate,0
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate,0
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High,0
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High,0
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low,0


## 2. Verify Schema Loading & Integrity Checks
We connect to the new SQLite database and verify the tables and row counts loaded.

In [5]:
# Connect to database
conn = sqlite3.connect(db_path)

# Query loaded tables
tables_df = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tables in Database:")
print(tables_df)

Tables in Database:
                    name
0        sqlite_sequence
1               dim_fund
2               dim_date
3               fact_nav
4      fact_transactions
5       fact_performance
6               fact_aum
7    monthly_sip_inflows
8       category_inflows
9   industry_folio_count
10    portfolio_holdings
11     benchmark_indices


In [6]:
# Check row counts for fact and dimension tables
row_counts = {}
for t in tables_df['name']:
    if t == 'sqlite_sequence': continue
    count = pd.read_sql(f"SELECT COUNT(*) as count FROM {t}", conn).iloc[0]['count']
    row_counts[t] = count

print("Database Row Counts:")
for tbl, cnt in row_counts.items():
    print(f"  {tbl:<22}: {cnt} rows")
conn.close()

Database Row Counts:
  dim_fund              : 40 rows
  dim_date              : 1826 rows
  fact_nav              : 64320 rows
  fact_transactions     : 32778 rows
  fact_performance      : 40 rows
  fact_aum              : 90 rows
  monthly_sip_inflows   : 48 rows
  category_inflows      : 144 rows
  industry_folio_count  : 21 rows
  portfolio_holdings    : 322 rows
  benchmark_indices     : 8050 rows


## 3. Findings
1. **NAV History Cleaning**: Reindexed the dates to cover full daily calendar date ranges, and forward-filled (`ffill`) missing NAV for weekends and market holidays. NAV values were validated to be strictly greater than 0.
2. **Transaction Cleaning**: Standardized the transaction type field to `SIP`, `Lumpsum`, or `Redemption`. KYC status was normalized to the enum values `Verified` or `Pending`.
3. **Performance Anomalies**: Found that standard deviation and returns are successfully loaded. Liquid funds with extremely low std deviation were flagged as `is_anomalous`.